# 04 – ML Forecasting

Build the feature matrix, train XGBoost / LightGBM / Random Forest models with time-series CV, and explain predictions with SHAP values.

**Covers:** Feature engineering pipeline, TimeSeriesSplit CV, directional accuracy, Signal Sharpe, SHAP summary plots.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from config.settings import DEFAULT_START_DATE, DEFAULT_END_DATE, FRED_API_KEY, DATA_DIR, TARGET_HORIZON
from src.data.fetcher import fetch_all_data

df = fetch_all_data(
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
    api_key=FRED_API_KEY,
    cache_dir=DATA_DIR,
)
print(f'Data: {df.shape}')

In [ ]:
from src.features.engineering import build_feature_matrix

X, y = build_feature_matrix(df, target_horizon=TARGET_HORIZON)
target_col = [c for c in y.columns if 'return' in c][0]
print(f'X shape: {X.shape}  |  target: {target_col}')
X.head(3)

In [ ]:
from src.models.ml_models import train_and_evaluate

results = {}
for model_type in ['xgboost', 'lightgbm', 'random_forest']:
    print(f'Training {model_type} ...')
    res = train_and_evaluate(X, y[target_col], model_type=model_type, n_splits=5)
    results[model_type] = res
    print(f'  {model_type}: {res["mean_metrics"]}')

In [ ]:
# Metrics comparison table
metrics_df = pd.DataFrame({k: v['mean_metrics'] for k, v in results.items()}).T
metrics_df.round(4)

In [ ]:
# Best model forecast vs actual
best_model = 'xgboost'
oof = results[best_model]['oof_predictions']
valid_mask = ~np.isnan(oof)

from src.visualization.plots import plot_forecast_vs_actual
fig = plot_forecast_vs_actual(y[target_col].values[valid_mask], oof[valid_mask],
                               title=f'{best_model} – OOF Forecast vs Actual')
fig.show()

In [ ]:
# SHAP feature importance
from src.models.ml_models import compute_shap_values
from src.visualization.plots import plot_shap_summary

shap_vals = compute_shap_values(results[best_model]['model'], X, model_type=best_model)
fig = plot_shap_summary(shap_vals, X, max_display=20)
fig.show()